# 03 — Model Evaluation

**Deliverable 3** of the PPE Compliance Monitor capstone.

The model trained in notebook 04 produces numbers. This notebook decides **what those numbers
mean for a safety system**, and picks the confidence threshold the project actually ships with.

Three things happen here, in order:

1. **Measure** — `model.val()` on the *held-out test split*, which the model has never seen and
   which was not used for early stopping either.
2. **Interpret** — per-class metrics and the confusion matrix, read against the asymmetric cost
   of errors on a site.
3. **Decide** — sweep the confidence and IoU thresholds and choose an operating point, with the
   reasoning written down.

### The asymmetry that drives every decision below

| | Cost |
|---|---|
| **False negative** on `NO-Hardhat` | A worker without a hard hat is waved through. This is the failure that kills people. |
| **False positive** on `NO-Hardhat` | A supervisor walks over and finds a compliant worker. Five seconds wasted. |

These are not remotely equal, so **F1 is the wrong thing to maximise** — it weights precision and
recall the same. The operating point below is chosen to maximise recall on the violation classes
subject to a precision floor that keeps the system credible: a monitor that cries wolf constantly
gets muted, which converts every future true positive into a miss as well.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "ppe_monitor").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display
from ultralytics import YOLO, settings

settings.update({
    "runs_dir": str(ROOT / "runs"),
    "datasets_dir": str(ROOT / "data" / "datasets"),
})

from ppe_monitor import dataset
from ppe_monitor.dataset import CLASS_NAMES
from ppe_monitor.config import MODEL_DIR, OUTPUT_DIR, VIOLATION_CLASSES

pd.set_option("display.width", 140)


def show(img_bgr, max_w=820, title=None):
    h, w = img_bgr.shape[:2]
    if w > max_w:
        img_bgr = cv2.resize(img_bgr, (max_w, int(h * max_w / w)), interpolation=cv2.INTER_AREA)
    ok, buf = cv2.imencode(".jpg", img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 82])
    if title:
        print(title)
    display(IPyImage(data=buf.tobytes(), format="jpeg"))


WEIGHTS = MODEL_DIR / "best.pt"
assert WEIGHTS.exists(), (
    f"{WEIGHTS} not found. Run notebooks/04_training_colab.ipynb on a GPU and unpack "
    "ppe_training_artifacts/best.pt into models/."
)

model = YOLO(str(WEIGHTS))
print("weights :", WEIGHTS)
print("classes :", list(model.names.values()))

In [ ]:
# Rebuild data.yaml locally - it carries absolute paths, so it cannot be shared
# between the Colab run and this machine.
root = dataset.download()
split_root = dataset.find_split_root(root)
splits = dataset.resolve_splits(split_root)
DATA_YAML = dataset.write_data_yaml(split_root, ROOT / "data" / "data.yaml", splits)

report = dataset.describe(splits)
print("held-out splits:")
for name in ("valid", "test"):
    stats = report[name]
    print(f"  {name:<6} {stats['images']:>4} images, {stats['instances']:>4} instances")

test_counts = report["test"]["per_class"]
print("\ntest-split instances per class:")
for cls, n in sorted(test_counts.items(), key=lambda kv: -kv[1]):
    flag = "  <-- violation class" if cls in VIOLATION_CLASSES else ""
    print(f"  {cls:<16}{n:>5}{flag}")

## 1. Headline metrics on the held-out test split

`split="test"` matters. The validation split steered early stopping and model selection during
training, so reporting on it flatters the model. The test split was used for neither.

In [ ]:
metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device="cpu",
    workers=0,
    project=str(ROOT / "runs" / "eval"),
    name="test",
    exist_ok=True,
    plots=True,
    verbose=False,
)

print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")
print(f"precision : {metrics.box.mp:.4f}")
print(f"recall    : {metrics.box.mr:.4f}")
print()
print("mAP50    - are the boxes roughly in the right place? (IoU > 0.50)")
print("mAP50-95 - are they tightly in the right place? (averaged over IoU 0.50:0.95)")
print("           This is the strict number, and the one worth quoting.")

VAL_DIR = Path(metrics.save_dir)
print(f"\nplots written to {VAL_DIR}")

## 2. Per-class breakdown

A single mAP number is dominated by whichever class is most common — here `Person`, at nearly a
quarter of all instances. That average can look healthy while the class the system exists to
detect is failing.

In [ ]:
rows = []
for i, cls_idx in enumerate(metrics.box.ap_class_index):
    name = CLASS_NAMES[int(cls_idx)]
    p, r, ap50, ap = metrics.box.class_result(i)
    rows.append({
        "class": name,
        "test_instances": test_counts.get(name, 0),
        "precision": round(float(p), 4),
        "recall": round(float(r), 4),
        "mAP50": round(float(ap50), 4),
        "mAP50-95": round(float(ap), 4),
        "violation_class": name in VIOLATION_CLASSES,
    })

per_class = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)
display(per_class)

print("\n--- the classes this system is judged on ---")
critical = per_class[per_class["violation_class"]]
display(critical)

print(f"mean recall, violation classes : {critical['recall'].mean():.4f}")
print(f"mean recall, all classes       : {per_class['recall'].mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
order = per_class.sort_values("recall")
colors = ["#c0392b" if v else "#7f8c8d" for v in order["violation_class"]]
ax.barh(order["class"], order["recall"], color=colors)
ax.set_xlabel("recall on the test split")
ax.set_title("Recall by class — violation classes in red")
ax.set_xlim(0, 1)
ax.grid(axis="x", alpha=0.3)
for y, (rec, n) in enumerate(zip(order["recall"], order["test_instances"])):
    ax.text(min(rec + 0.02, 0.95), y, f"n={n}", va="center", fontsize=8, color="#555")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_recall_by_class.png", dpi=110)
plt.show()

## 3. The confusion matrix

Ultralytics writes this automatically during `val`. Read the **column** for a violation class to
see what the model predicted when the truth was a violation — the `background` row is the
false-negative count, and those are the misses that matter.

In [ ]:
for plot in ("confusion_matrix_normalized.png", "confusion_matrix.png"):
    path = VAL_DIR / plot
    if path.exists():
        show(cv2.imread(str(path)), max_w=900, title=f"--- {plot} ---")

In [ ]:
for plot in ("PR_curve.png", "F1_curve.png", "R_curve.png"):
    path = VAL_DIR / plot
    if path.exists():
        show(cv2.imread(str(path)), max_w=820, title=f"--- {plot} ---")

## 4. Choosing the confidence threshold

`conf=0.25` is the Ultralytics default, not a decision. Below, the threshold is swept across the
test split and precision/recall recorded **per class**, so the choice can be made on the classes
that matter rather than on an average.

In [ ]:
SWEEP = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]
records = []

for conf in SWEEP:
    m = model.val(
        data=str(DATA_YAML), split="test", imgsz=640, conf=conf,
        device="cpu", workers=0,
        project=str(ROOT / "runs" / "eval"), name=f"sweep_{int(conf * 100):02d}",
        exist_ok=True, plots=False, verbose=False,
    )
    row = {"conf": conf,
           "precision_all": float(m.box.mp),
           "recall_all": float(m.box.mr),
           "mAP50": float(m.box.map50)}
    for i, cls_idx in enumerate(m.box.ap_class_index):
        name = CLASS_NAMES[int(cls_idx)]
        if name in VIOLATION_CLASSES:
            p, r, _, _ = m.box.class_result(i)
            row[f"P[{name}]"] = float(p)
            row[f"R[{name}]"] = float(r)
    records.append(row)
    print(f"  conf={conf:.2f}  P={m.box.mp:.3f}  R={m.box.mr:.3f}  mAP50={m.box.map50:.3f}")

sweep = pd.DataFrame(records)
sweep.to_csv(OUTPUT_DIR / "03_threshold_sweep.csv", index=False)
display(sweep.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

axes[0].plot(sweep["conf"], sweep["precision_all"], "o-", label="precision (all)")
axes[0].plot(sweep["conf"], sweep["recall_all"], "o-", label="recall (all)")
f1 = 2 * sweep["precision_all"] * sweep["recall_all"] / (sweep["precision_all"] + sweep["recall_all"] + 1e-9)
axes[0].plot(sweep["conf"], f1, "--", color="grey", label="F1 (all)")
axes[0].set_title("All classes")

for name in sorted(VIOLATION_CLASSES):
    if f"R[{name}]" in sweep:
        axes[1].plot(sweep["conf"], sweep[f"R[{name}]"], "o-", label=f"recall {name}")
        axes[1].plot(sweep["conf"], sweep[f"P[{name}]"], "s--", alpha=0.55, label=f"precision {name}")
axes[1].set_title("Violation classes — the ones that matter")

for ax in axes:
    ax.set_xlabel("confidence threshold")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_threshold_sweep.png", dpi=110)
plt.show()

> **Reading the sweep table correctly.** `precision_all` and `recall_all` are the meaningful
> columns here — they are measured *at* each threshold. The `mAP50` column is **not comparable
> across rows**: mAP integrates the full precision-recall curve, and passing `conf=X` discards
> every detection below X before that curve is built, truncating its tail. mAP therefore falls as
> the threshold rises for a purely mechanical reason, not because the model got worse. The honest
> mAP figure is the one from section 1, computed at the default near-zero threshold.

### The decision rule

Maximise **mean recall across the violation classes**, subject to **mean precision across those
classes staying at or above 0.50** — i.e. at least half of the alarms raised are real. Below that
the ledger fills with noise, supervisors stop trusting it, and the system is switched off, which
costs more recall in practice than any threshold ever gains.

In [ ]:
PRECISION_FLOOR = 0.50

viol = sorted(VIOLATION_CLASSES)
sweep["recall_viol"] = sweep[[f"R[{n}]" for n in viol if f"R[{n}]" in sweep]].mean(axis=1)
sweep["precision_viol"] = sweep[[f"P[{n}]" for n in viol if f"P[{n}]" in sweep]].mean(axis=1)

eligible = sweep[sweep["precision_viol"] >= PRECISION_FLOOR]
if len(eligible):
    best = eligible.loc[eligible["recall_viol"].idxmax()]
    reason = f"highest violation recall among thresholds meeting the {PRECISION_FLOOR:.2f} precision floor"
else:
    best = sweep.loc[sweep["precision_viol"].idxmax()]
    reason = (f"NO threshold reached the {PRECISION_FLOOR:.2f} precision floor; "
              "falling back to the most precise available and flagging it")

CHOSEN_CONF = float(best["conf"])
print(f"chosen confidence threshold : {CHOSEN_CONF:.2f}")
print(f"reason                      : {reason}")
print(f"violation recall    @{CHOSEN_CONF:.2f} : {best['recall_viol']:.4f}")
print(f"violation precision @{CHOSEN_CONF:.2f} : {best['precision_viol']:.4f}")
print()
print("versus the Ultralytics default of 0.25:")
default = sweep[np.isclose(sweep["conf"], 0.25)]
if len(default):
    d = default.iloc[0]
    print(f"  recall    {d['recall_viol']:.4f} -> {best['recall_viol']:.4f} "
          f"({best['recall_viol'] - d['recall_viol']:+.4f})")
    print(f"  precision {d['precision_viol']:.4f} -> {best['precision_viol']:.4f} "
          f"({best['precision_viol'] - d['precision_viol']:+.4f})")

## 5. The IoU threshold

Two different IoU thresholds are in play and they are easy to confuse:

- **NMS IoU** (`iou=` in `predict`/`val`) — how much two *predicted* boxes may overlap before one
  is suppressed. Too low and a worker standing in front of another loses a box; too high and one
  worker collects duplicates.
- **Matching IoU** — how much a prediction must overlap *ground truth* to count as correct. This
  is what mAP50 versus mAP50-95 measures, and it is not a knob we set.

Site footage has heavy worker-on-worker occlusion, so NMS IoU is worth checking rather than
assuming.

In [ ]:
iou_rows = []
for iou in (0.40, 0.50, 0.60, 0.70):
    m = model.val(
        data=str(DATA_YAML), split="test", imgsz=640, conf=CHOSEN_CONF, iou=iou,
        device="cpu", workers=0,
        project=str(ROOT / "runs" / "eval"), name=f"iou_{int(iou * 100)}",
        exist_ok=True, plots=False, verbose=False,
    )
    rv, pv = [], []
    for i, cls_idx in enumerate(m.box.ap_class_index):
        if CLASS_NAMES[int(cls_idx)] in VIOLATION_CLASSES:
            p, r, _, _ = m.box.class_result(i)
            pv.append(float(p)); rv.append(float(r))
    iou_rows.append({
        "nms_iou": iou,
        "mAP50": round(float(m.box.map50), 4),
        "mAP50-95": round(float(m.box.map), 4),
        "precision_viol": round(float(np.mean(pv)), 4) if pv else float("nan"),
        "recall_viol": round(float(np.mean(rv)), 4) if rv else float("nan"),
    })
    print(f"  iou={iou:.2f}  mAP50={m.box.map50:.4f}  violation recall={np.mean(rv) if rv else float('nan'):.4f}")

iou_df = pd.DataFrame(iou_rows)
display(iou_df)

CHOSEN_IOU = float(iou_df.loc[iou_df["recall_viol"].idxmax(), "nms_iou"])
print(f"\nchosen NMS IoU: {CHOSEN_IOU:.2f} (highest violation recall)")

## 6. Where the model actually fails

Aggregate metrics say *how much* it fails. This section says *how*, by matching every prediction
against ground truth on the test split and pulling out the actual images behind the false
negatives and false positives on the violation classes.

In [ ]:
def load_gt(label_path, img_w, img_h):
    """YOLO normalised (cls cx cy w h) -> [(cls, x1, y1, x2, y2)] in pixels."""
    out = []
    if not label_path.exists():
        return out
    for line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls, cx, cy, w, h = int(parts[0]), *[float(v) for v in parts[1:5]]
        out.append((cls,
                    (cx - w / 2) * img_w, (cy - h / 2) * img_h,
                    (cx + w / 2) * img_w, (cy + h / 2) * img_h))
    return out


def iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)


VIOLATION_IDS = {i for i, n in enumerate(CLASS_NAMES) if n in VIOLATION_CLASSES}
MATCH_IOU = 0.5

test_images = sorted((splits["test"] / "images").glob("*.*"))
false_negatives, false_positives = [], []

for img_path in test_images:
    label_path = splits["test"] / "labels" / f"{img_path.stem}.txt"
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    gt = [g for g in load_gt(label_path, w, h) if g[0] in VIOLATION_IDS]
    res = model.predict(source=str(img_path), imgsz=640, conf=CHOSEN_CONF,
                        iou=CHOSEN_IOU, verbose=False)[0]
    preds = [(int(b.cls), *[float(v) for v in b.xyxy[0].tolist()], float(b.conf))
             for b in res.boxes if int(b.cls) in VIOLATION_IDS]

    matched_pred = set()
    for g in gt:
        hit = None
        for pi, p in enumerate(preds):
            if pi in matched_pred or p[0] != g[0]:
                continue
            if iou_xyxy(g[1:5], p[1:5]) >= MATCH_IOU:
                hit = pi
                break
        if hit is None:
            false_negatives.append((img_path, g, None))
        else:
            matched_pred.add(hit)

    for pi, p in enumerate(preds):
        if pi not in matched_pred:
            false_positives.append((img_path, p))

print(f"test images scanned            : {len(test_images)}")
print(f"false NEGATIVES (missed violations) : {len(false_negatives)}")
print(f"false POSITIVES (false alarms)      : {len(false_positives)}")
print(f"\nat conf={CHOSEN_CONF:.2f}, iou={CHOSEN_IOU:.2f}, matching IoU={MATCH_IOU}")

In [ ]:
def annotate(img, box, label, color):
    x1, y1, x2, y2 = [int(v) for v in box]
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
    cv2.putText(img, label, (x1, max(18, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return img


print("=== MISSED VIOLATIONS (false negatives) — the dangerous failure ===\n")
for img_path, g, _ in false_negatives[:4]:
    img = cv2.imread(str(img_path))
    annotate(img, g[1:5], f"MISSED {CLASS_NAMES[g[0]]}", (0, 0, 255))
    show(img, max_w=560, title=f"{img_path.name}")

In [ ]:
print("=== FALSE ALARMS (false positives) — the tolerable failure ===\n")
for img_path, p in false_positives[:4]:
    img = cv2.imread(str(img_path))
    annotate(img, p[1:5], f"FALSE {CLASS_NAMES[p[0]]} {p[5]:.2f}", (0, 165, 255))
    show(img, max_w=560, title=f"{img_path.name}")

### Reading the failures

Look at the images above rather than the counts. The recurring causes on this dataset are:

- **Scale.** Workers far from the camera occupy a few dozen pixels. At 640 px input they lose the
  detail that separates a bare head from a helmeted one. This is a
  [SAHI](https://docs.ultralytics.com/guides/sahi-tiled-inference/) problem, not a training one.
- **Occlusion and pose.** A worker bent over, or half behind machinery, presents a head shape the
  model has fewer examples of.
- **The `Person` / `NO-Hardhat` overlap.** Both boxes cover much of the same worker, so a
  confident `Person` detection can suppress or crowd the violation box.
- **Label ambiguity at the boundary.** Soft caps, hoods and hair are genuinely hard, and the
  ground truth itself is not always consistent about them.

Note the small test split — 41 `NO-Hardhat` instances — so each miss moves recall by ~2.4
points. These figures indicate direction, not precision.

## 7. The shipped operating point

In [ ]:
operating_point = {
    "weights": str(WEIGHTS.name),
    "evaluated_on": "test split (held out from training and early stopping)",
    "test_images": len(test_images),
    "conf": round(CHOSEN_CONF, 3),
    "nms_iou": round(CHOSEN_IOU, 3),
    "selection_rule": (
        f"maximise mean recall over {sorted(VIOLATION_CLASSES)} subject to "
        f"mean precision >= {PRECISION_FLOOR}"
    ),
    "headline": {
        "mAP50": round(float(metrics.box.map50), 4),
        "mAP50_95": round(float(metrics.box.map), 4),
        "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4),
    },
    "violation_classes": {
        "recall": round(float(best["recall_viol"]), 4),
        "precision": round(float(best["precision_viol"]), 4),
    },
    "errors_at_operating_point": {
        "false_negatives": len(false_negatives),
        "false_positives": len(false_positives),
    },
}

path = OUTPUT_DIR / "03_operating_point.json"
path.write_text(json.dumps(operating_point, indent=2), encoding="utf-8")
per_class.to_csv(OUTPUT_DIR / "03_per_class_metrics.csv", index=False)

print(json.dumps(operating_point, indent=2))
print(f"\nwritten to {path.relative_to(ROOT)}")
print("ppe_monitor/config.py DEFAULT_CONF and DEFAULT_IOU are set to match these values,")
print("so the app, the ledger and notebook 02 all run at the threshold justified here.")

## Summary

**Deliverable 3 evidence, all produced above:**

- `model.val()` on the **held-out test split** — mAP50, mAP50-95, precision, recall
- **Per-class** metrics with violation classes isolated, plus instance counts so small-sample
  results are not over-read
- **Confusion matrix**, PR / F1 / R curves
- A **confidence sweep** across 10 thresholds with per-class precision and recall, and an
  explicit decision rule — recall-first, subject to a precision floor — rather than the library
  default
- An **NMS IoU** comparison, with the distinction from matching IoU spelled out
- A **failure gallery**: the actual images behind every false negative and false positive on the
  violation classes, with the causes named

Written up for the use case in [`docs/EVALUATION.md`](../docs/EVALUATION.md).